# 🧬 H5N1 Phylogeny × Embedding Comparison
**Interactive visual comparison of IQ-TREE topology vs UMAP / t-SNE / Tucker / SVD embeddings**

Pipeline: `FAMSA → IQ-TREE` vs `FAMSA → distance matrix → dimensionality reduction`

---
### How to use this notebook
1. **Run Cell 1** — set your pipeline root path  
2. **Run Cell 2** — imports  
3. **Run Cell 3** — helper functions (run once)  
4. **Run Cell 4** — pick a segment and embedding, then interact with the widgets


In [ ]:
import subprocess
subprocess.run(["pip", "install", "ipywidgets"], check=True)
import ipywidgets as widgets

In [ ]:

# ── Configuration — edit this cell to match your environment ─────────────────
PIPELINE = "/data/users/ltucker/influenzaData/H5N1_pipeline"

# Segment number → influenza segment name
SEGMENT_NAMES = {
    1: "PB2", 2: "PB1", 3: "PA", 4: "HA",
    5: "NP",  6: "NA",  7: "MP", 8: "NS"
}

# Colour palette (one per clade)
PALETTE = [
    "#00d4ff","#ff6b6b","#ffd93d","#6bcb77","#ff922b",
    "#cc5de8","#74c0fc","#f06595","#a9e34b","#ff8787",
    "#4dabf7","#63e6be","#ffa94d","#da77f2","#66d9e8",
    "#f783ac","#8ce99a","#ffe066","#a5d8ff","#ffb2b2",
]

print("✓ Config loaded — pipeline root:", PIPELINE)


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Ellipse
from scipy.spatial.distance import pdist, squareform
from scipy.stats import spearmanr
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.cluster import KMeans

import ipywidgets as widgets
from IPython.display import display, clear_output

try:
    from Bio import Phylo
    HAS_BIO = True
    print("✓ biopython")
except ImportError:
    HAS_BIO = False
    print("✗ biopython not found — install with: mamba install -c conda-forge biopython")

try:
    import toytree
    HAS_TOYTREE = True
    print(f"✓ toytree {toytree.__version__}")
except ImportError:
    HAS_TOYTREE = False
    print("✗ toytree not found — install with: mamba install -c conda-forge toytree")

print("✓ All other imports OK")


In [ ]:
# ════════════════════════════════════════════════════════
#  HELPER FUNCTIONS  —  run this cell once
# ════════════════════════════════════════════════════════

# ── Discover available embeddings for a segment ──────────────────────────────
def get_available_embeddings(segment_num: int) -> dict:
    """Returns dict of {display_label: parquet_path} for a segment."""
    seg_name = SEGMENT_NAMES[segment_num]
    base = Path(PIPELINE)
    found = {}

    # 1. Parameter sweep UMAP / t-SNE
    sweep_dir = base / "output/famsa_per_segment_analysis/embedding_param_sweep/embeddings" / seg_name
    if sweep_dir.exists():
        for p in sorted(sweep_dir.glob("*.parquet")):
            label = f"sweep/{p.stem}"
            found[label] = str(p)

    # 2. Standard embeddings (tuned_v2 > tuned > base)
    for version in ["embeddings_tuned_v2", "embeddings_tuned", "embeddings"]:
        emb_path = base / "output/famsa_per_segment_analysis" / version / f"embeddings_{seg_name}.parquet"
        if emb_path.exists():
            found[f"{version}/embeddings_{seg_name}"] = str(emb_path)

    # 3. Tucker / SVD factor embeddings
    for dr_file in sorted((base / "output/famsa_tensor_analysis/dimensionality_reduction").glob("*.parquet")):
        found[f"tensor/{dr_file.stem}"] = str(dr_file)

    return found


# ── Load embedding parquet ────────────────────────────────────────────────────
def load_embedding(path: str) -> pd.DataFrame:
    df = pd.read_parquet(path)
    df.columns = [str(c).lower().strip() for c in df.columns]
    # normalise id column → "name"
    for candidate in ["sequence_id","seq_id","id","header","accession","sample_id","index"]:
        if candidate in df.columns and "name" not in df.columns:
            df = df.rename(columns={candidate: "name"})
            break
    if "name" not in df.columns:
        df = df.reset_index().rename(columns={"index": "name"})
    coord_cols = [c for c in df.columns if c != "name"][:2]
    df = df.rename(columns={coord_cols[0]: "x", coord_cols[1]: "y"})
    df["name"] = df["name"].astype(str).str.strip()
    df["x"] = pd.to_numeric(df["x"], errors="coerce")
    df["y"] = pd.to_numeric(df["y"], errors="coerce")
    return df[["name","x","y"]].dropna().reset_index(drop=True)


# ── Load IQ-TREE treefile ────────────────────────────────────────────────────
def load_tree(segment_num: int):
    path = Path(PIPELINE) / f"output/iqtree/segment_{segment_num}.treefile"
    if not path.exists():
        print(f"  ✗ Tree not found: {path}")
        return None
    if HAS_BIO:
        tree = Phylo.read(str(path), "newick")
        tips = tree.get_terminals()
        print(f"  ✓ Tree loaded — {len(tips)} tips")
        return tree
    print("  ✗ biopython required to load tree")
    return None


# ── Also try IQ-TREE .mldist file (much faster than recomputing distances) ───
def load_mldist(segment_num: int) -> pd.DataFrame | None:
    path = Path(PIPELINE) / f"output/iqtree/segment_{segment_num}.mldist"
    if not path.exists():
        return None
    lines = path.read_text().strip().split("\n")
    n = int(lines[0].strip())
    names, rows = [], []
    for line in lines[1:n+1]:
        parts = line.split()
        names.append(parts[0])
        rows.append([float(x) for x in parts[1:]])
    df = pd.DataFrame(rows, index=names, columns=names)
    print(f"  ✓ .mldist loaded — {n}×{n} distance matrix")
    return df


# ── Assign clades from distance matrix ───────────────────────────────────────
def assign_clades(dist_df: pd.DataFrame, n_clades: int) -> dict:
    """Hierarchical clustering on patristic distances → {name: clade_id}."""
    names = list(dist_df.index)
    dist_arr = dist_df.values.astype(float)
    np.fill_diagonal(dist_arr, 0)
    Z = linkage(squareform(dist_arr), method="average")
    labels = fcluster(Z, n_clades, criterion="maxclust")
    return {name: int(labels[i]) - 1 for i, name in enumerate(names)}


# ── Compute stats ─────────────────────────────────────────────────────────────
def compute_stats(emb_df: pd.DataFrame, clade_map: dict, dist_df: pd.DataFrame) -> dict:
    shared = [n for n in emb_df["name"] if n in clade_map and n in dist_df.index]
    if len(shared) < 4:
        return {}
    sub_emb = emb_df.set_index("name").loc[shared, ["x","y"]].values
    sub_dist = dist_df.loc[shared, shared].values.astype(float)
    umap_dist = squareform(pdist(sub_emb))
    idx = np.triu_indices(len(shared), k=1)
    r, p = spearmanr(sub_dist[idx], umap_dist[idx])
    tree_labels = [clade_map[n] for n in shared]
    n_clades = len(set(tree_labels))
    km = KMeans(n_clusters=max(2, n_clades), random_state=42, n_init=10)
    umap_labels = km.fit_predict(sub_emb)
    ari = adjusted_rand_score(tree_labels, umap_labels)
    nmi = normalized_mutual_info_score(tree_labels, umap_labels)
    return {"mantel_r": r, "mantel_p": p, "ARI": ari, "NMI": nmi, "n_shared": len(shared)}


# ── Draw tree panel ───────────────────────────────────────────────────────────
BG = "#0a0e1a"; GRID = "#1a2a3a"; TEXT = "#c8d8f0"; DIM = "#3a5070"

def style_ax(ax):
    ax.set_facecolor(BG)
    ax.tick_params(colors=DIM, labelsize=8)
    for sp in ax.spines.values(): sp.set_edgecolor(GRID)

def draw_tree(ax, tree, clade_map, highlight_name=None, show_labels=False):
    style_ax(ax)
    if tree is None:
        ax.text(0.5,0.5,"Tree not loaded", ha="center", va="center",
                color=DIM, transform=ax.transAxes)
        return

    tips = tree.get_terminals()
    leaf_y = {t.name: i for i, t in enumerate(tips)}

    def get_x(clade, px=0.0):
        clade._x = px + (clade.branch_length or 0.0)
        for c in clade.clades: get_x(c, clade._x)

    def get_y(clade):
        if not clade.clades:
            clade._y = leaf_y.get(clade.name, 0)
        else:
            for c in clade.clades: get_y(c)
            clade._y = np.mean([c._y for c in clade.clades])

    get_x(tree.root); get_y(tree.root)

    def draw(clade):
        for child in clade.clades:
            def first_tip(c):
                return c.name if not c.clades else first_tip(c.clades[0])
            tip_name = first_tip(child)
            cid = clade_map.get(tip_name, -1)
            color = PALETTE[cid % len(PALETTE)] if cid >= 0 else DIM
            hi = highlight_name and (clade_map.get(highlight_name,-1) == cid)
            alpha = 1.0 if (not highlight_name or hi) else 0.15
            lw = 1.8 if hi else 0.9
            ax.plot([clade._x, clade._x, child._x],
                    [clade._y, child._y, child._y],
                    color=color, lw=lw, alpha=alpha, solid_capstyle="round")
            draw(child)
    draw(tree.root)

    for t in tips:
        cid = clade_map.get(t.name, -1)
        color = PALETTE[cid % len(PALETTE)] if cid >= 0 else DIM
        hi = highlight_name and (clade_map.get(highlight_name,-1) == cid)
        alpha = 1.0 if (not highlight_name or hi) else 0.15
        ax.scatter(t._x, leaf_y[t.name], color=color, s=10, zorder=4,
                   alpha=alpha, linewidths=0)
        if show_labels or t.name == highlight_name:
            ax.text(t._x + 0.0002, leaf_y[t.name], t.name,
                    fontsize=6, color=color, va="center", alpha=alpha)

    ax.set_xlabel("Branch length", color=DIM, fontsize=9)
    ax.set_yticks([])
    ax.invert_yaxis()
    ax.set_title(f"IQ-TREE topology", color=TEXT, fontsize=10, pad=6)


# ── Draw embedding panel ──────────────────────────────────────────────────────
def draw_embedding(ax, emb_df, clade_map, embed_label="",
                   highlight_name=None, show_labels=False, show_ellipses=True):
    style_ax(ax)
    if emb_df is None or emb_df.empty:
        ax.text(0.5,0.5,"Embedding not loaded", ha="center", va="center",
                color=DIM, transform=ax.transAxes)
        return

    # scatter
    for _, row in emb_df.iterrows():
        cid = clade_map.get(row["name"], -1)
        color = PALETTE[cid % len(PALETTE)] if cid >= 0 else "#333"
        hi = highlight_name and (clade_map.get(highlight_name,-1) == cid)
        exact = row["name"] == highlight_name
        alpha = 1.0 if (not highlight_name or hi) else 0.12
        s = 55 if exact else (18 if hi else 12)
        ax.scatter(row["x"], row["y"], c=color, s=s, alpha=alpha,
                   linewidths=0.4 if exact else 0, edgecolors="white", zorder=3)
        if show_labels or exact:
            ax.text(row["x"]+0.02, row["y"], row["name"],
                    fontsize=6, color=color, va="center", alpha=alpha)

    # ellipses per clade
    if show_ellipses:
        by_clade = {}
        for _, row in emb_df.iterrows():
            cid = clade_map.get(row["name"], -1)
            if cid < 0: continue
            by_clade.setdefault(cid, []).append([row["x"], row["y"]])
        for cid, pts in by_clade.items():
            if len(pts) < 3: continue
            pts = np.array(pts)
            cx, cy = pts.mean(0)
            rx = pts[:,0].std()*2 + 0.05
            ry = pts[:,1].std()*2 + 0.05
            hi = highlight_name and clade_map.get(highlight_name,-1) == cid
            alpha = 0.55 if (not highlight_name or hi) else 0.08
            ell = Ellipse((cx,cy), rx*2, ry*2, edgecolor=PALETTE[cid % len(PALETTE)],
                          facecolor="none", lw=1.2, linestyle="--", alpha=alpha, zorder=2)
            ax.add_patch(ell)

    ax.set_xlabel("Dim 1", color=DIM, fontsize=9)
    ax.set_ylabel("Dim 2", color=DIM, fontsize=9)
    ax.set_title(embed_label, color=TEXT, fontsize=10, pad=6)


# ── Stats panel ───────────────────────────────────────────────────────────────
def draw_stats(ax, stats, n_clades, seg_name, n_matched, n_total):
    ax.set_facecolor(BG); ax.axis("off")
    lines = [
        ("Segment",          seg_name),
        ("Clades (tree cut)", str(n_clades)),
        ("Sequences matched", f"{n_matched} / {n_total}"),
    ]
    if stats:
        lines += [
            ("Mantel r",   f"{stats['mantel_r']:.3f}"),
            ("Mantel p",   f"{stats['mantel_p']:.4f}"),
            ("ARI",        f"{stats['ARI']:.3f}"),
            ("NMI",        f"{stats['NMI']:.3f}"),
        ]
    y = 0.97
    ax.text(0.5, y, "Summary", ha="center", va="top", color=TEXT,
            fontsize=11, fontweight="bold", transform=ax.transAxes)
    y -= 0.10
    for label, val in lines:
        ax.text(0.05, y, label, color=DIM, fontsize=9, transform=ax.transAxes, va="top")
        ax.text(0.95, y, val, color="#00d4ff", fontsize=9, fontweight="bold",
                ha="right", transform=ax.transAxes, va="top")
        y -= 0.09

    if stats and "mantel_r" in stats:
        r = stats["mantel_r"]
        msg  = "Strong agreement ✓" if r > 0.7 else "Moderate agreement" if r > 0.4 else "Weak agreement ✗"
        col  = "#6bcb77" if r > 0.7 else "#ffd93d" if r > 0.4 else "#ff6b6b"
        ax.text(0.5, 0.06, msg, ha="center", va="bottom", color=col,
                fontsize=9, fontweight="bold", transform=ax.transAxes)

    # legend
    pass  # legend drawn separately


print("✓ Helper functions ready — proceed to Cell 4")


In [ ]:
# ════════════════════════════════════════════════════════
#  INTERACTIVE COMPARISON WIDGET
#  Run this cell — controls appear below
# ════════════════════════════════════════════════════════

# ── State ─────────────────────────────────────────────────────────────────────
_state = {"tree": None, "dist_df": None, "clade_map": {}, "emb_df": None,
          "seg": None, "embed_label": ""}

# ── Widgets ───────────────────────────────────────────────────────────────────
w_seg = widgets.Dropdown(
    options=[(f"Seg {k} — {v}", k) for k, v in SEGMENT_NAMES.items()],
    value=4, description="Segment:", style={"description_width": "80px"},
    layout=widgets.Layout(width="260px")
)
w_embed = widgets.Dropdown(
    options=[], description="Embedding:", style={"description_width": "80px"},
    layout=widgets.Layout(width="420px")
)
w_clades = widgets.IntSlider(
    value=8, min=2, max=20, step=1, description="Clades:",
    style={"description_width": "80px"}, layout=widgets.Layout(width="320px"),
    continuous_update=False
)
w_labels = widgets.Checkbox(value=False, description="Show labels",
                             layout=widgets.Layout(width="140px"))
w_ellipses = widgets.Checkbox(value=True, description="Show ellipses",
                               layout=widgets.Layout(width="150px"))
w_stats = widgets.Checkbox(value=True, description="Compute stats",
                            layout=widgets.Layout(width="150px"))
w_search = widgets.Text(
    placeholder="Search sequence name…", description="Highlight:",
    style={"description_width": "70px"}, layout=widgets.Layout(width="340px")
)
w_status = widgets.HTML(value="<i style='color:#3a5070'>Load a segment to begin.</i>")
out = widgets.Output()

# ── Load segment (tree + dist matrix) ────────────────────────────────────────
def on_segment_change(change):
    seg = change["new"]
    _state["seg"] = seg
    seg_name = SEGMENT_NAMES[seg]
    w_status.value = f"<i style='color:#ffd93d'>Loading segment {seg} ({seg_name})…</i>"

    _state["tree"]    = load_tree(seg)
    _state["dist_df"] = load_mldist(seg)

    # populate embedding dropdown
    embeddings = get_available_embeddings(seg)
    w_embed.options = list(embeddings.items())  # (label, path) tuples
    if embeddings:
        w_embed.value = list(embeddings.values())[0]

    n_tips = len(_state["tree"].get_terminals()) if _state["tree"] else "?"
    w_status.value = (f"<b style='color:#6bcb77'>✓ Segment {seg} ({seg_name}) loaded</b> "
                      f"— {n_tips} tips | "
                      f"{len(embeddings)} embeddings available")
    refresh_plot()

# ── Load embedding ────────────────────────────────────────────────────────────
def on_embed_change(change):
    path = change["new"]
    if not path: return
    try:
        _state["emb_df"] = load_embedding(path)
        _state["embed_label"] = w_embed.label if hasattr(w_embed, "label") else Path(path).stem
        refresh_plot()
    except Exception as e:
        w_status.value = f"<span style='color:#ff6b6b'>✗ Could not load embedding: {e}</span>"

# ── Main plot refresh ─────────────────────────────────────────────────────────
def refresh_plot(*args):
    with out:
        clear_output(wait=True)

        tree     = _state["tree"]
        dist_df  = _state["dist_df"]
        emb_df   = _state["emb_df"]
        seg      = _state["seg"]

        if seg is None:
            print("Select a segment above to begin.")
            return

        seg_name = SEGMENT_NAMES[seg]
        n_clades = w_clades.value
        highlight = w_search.value.strip() or None

        # recompute clade map
        if dist_df is not None:
            clade_map = assign_clades(dist_df, n_clades)
        else:
            clade_map = {}

        _state["clade_map"] = clade_map

        # stats
        stats = {}
        if w_stats.value and emb_df is not None and dist_df is not None and len(clade_map) > 0:
            stats = compute_stats(emb_df, clade_map, dist_df)

        n_matched = sum(1 for n in (emb_df["name"] if emb_df is not None else [])
                        if n in clade_map)
        n_total   = len(emb_df) if emb_df is not None else 0

        # ── Figure ────────────────────────────────────────────────────────────
        fig = plt.figure(figsize=(18, 8), facecolor=BG)
        fig.suptitle(
            f"H5N1  ·  Segment {seg} ({seg_name})  ·  {n_clades} clades  ·  "
            f"{Path(w_embed.value).stem if w_embed.value else ''}",
            color=TEXT, fontsize=12, fontweight="bold", y=0.99
        )
        gs = fig.add_gridspec(1, 3, width_ratios=[2.2, 2.2, 1],
                              wspace=0.28, left=0.04, right=0.97,
                              top=0.93, bottom=0.07)

        ax_tree  = fig.add_subplot(gs[0])
        ax_emb   = fig.add_subplot(gs[1])
        ax_stats = fig.add_subplot(gs[2])

        draw_tree(ax_tree, tree, clade_map,
                  highlight_name=highlight, show_labels=w_labels.value)
        draw_embedding(ax_emb, emb_df, clade_map,
                       embed_label=Path(w_embed.value).stem if w_embed.value else "",
                       highlight_name=highlight,
                       show_labels=w_labels.value,
                       show_ellipses=w_ellipses.value)
        draw_stats(ax_stats, stats, n_clades, seg_name, n_matched, n_total)

        # shared legend below both plots
        unique_clades = sorted(set(clade_map.values()))
        handles = [mpatches.Patch(facecolor=PALETTE[c % len(PALETTE)],
                                  label=f"Clade {c+1}", edgecolor="#1a2a3a")
                   for c in unique_clades]
        fig.legend(handles=handles, loc="lower center",
                   ncol=min(len(handles), 10),
                   frameon=False, labelcolor=TEXT, fontsize=8,
                   bbox_to_anchor=(0.45, -0.01))

        plt.savefig("/tmp/_phylo_preview.png", dpi=130, bbox_inches="tight",
                    facecolor=BG)
        plt.show()
        plt.close()

# ── Wire up observers ─────────────────────────────────────────────────────────
w_seg.observe(on_segment_change, names="value")
w_embed.observe(on_embed_change, names="value")
w_clades.observe(refresh_plot, names="value")
w_labels.observe(refresh_plot, names="value")
w_ellipses.observe(refresh_plot, names="value")
w_stats.observe(refresh_plot, names="value")
w_search.observe(refresh_plot, names="value")

# ── Layout and display ────────────────────────────────────────────────────────
row1 = widgets.HBox([w_seg, w_embed])
row2 = widgets.HBox([w_clades, w_labels, w_ellipses, w_stats])
row3 = widgets.HBox([w_search])
panel = widgets.VBox(
    [row1, row2, row3, w_status, out],
    layout=widgets.Layout(border="1px solid #1e3a5a", padding="10px",
                          background_color="#0a0e1a")
)
display(panel)

# trigger initial load
w_seg.value = 4


In [ ]:
display(out)

---
### Export

In [ ]:
seg_num  = 4   # HA
n_clades = 8

tree      = load_tree(seg_num)
dist_df   = load_mldist(seg_num)
clade_map = assign_clades(dist_df, n_clades)

embeddings = get_available_embeddings(seg_num)
print("Available embeddings:")
for k, v in embeddings.items():
    print(f"  {k}")

emb_df = load_embedding(list(embeddings.values())[0])

fig, axes = plt.subplots(1, 2, figsize=(16, 7), facecolor=BG)
draw_tree(axes[0], tree, clade_map)
draw_embedding(axes[1], emb_df, clade_map)
plt.tight_layout()
plt.show()

In [ ]:
# ── Export current view to PDF ───────────────────────────────────────────────
# Change filename or format (.pdf / .png / .svg) as needed

seg      = _state["seg"] or 4
seg_name = SEGMENT_NAMES.get(seg, "unknown")
embed    = Path(w_embed.value).stem if w_embed.value else "embedding"
clade_map = _state["clade_map"]
n_clades  = w_clades.value

out_path = Path(PIPELINE) / f"output/phylo_umap_comparison/segment_{seg}_{seg_name}_{embed}_c{n_clades}.pdf"
out_path.parent.mkdir(parents=True, exist_ok=True)

fig = plt.figure(figsize=(20, 9), facecolor=BG)
fig.suptitle(f"H5N1 · Segment {seg} ({seg_name}) · {n_clades} clades · {embed}",
             color=TEXT, fontsize=13, fontweight="bold")
gs = fig.add_gridspec(1, 3, width_ratios=[2.2, 2.2, 1],
                      wspace=0.28, left=0.04, right=0.97, top=0.92, bottom=0.08)
ax_tree  = fig.add_subplot(gs[0])
ax_emb   = fig.add_subplot(gs[1])
ax_stats = fig.add_subplot(gs[2])

stats = compute_stats(_state["emb_df"], clade_map, _state["dist_df"]) \
        if _state["emb_df"] is not None and _state["dist_df"] is not None else {}

draw_tree(ax_tree, _state["tree"], clade_map, show_labels=w_labels.value)
draw_embedding(ax_emb, _state["emb_df"], clade_map, embed_label=embed,
               show_labels=w_labels.value, show_ellipses=w_ellipses.value)
draw_stats(ax_stats, stats, n_clades, seg_name,
           sum(1 for n in _state["emb_df"]["name"] if n in clade_map),
           len(_state["emb_df"]))

unique_clades = sorted(set(clade_map.values()))
handles = [mpatches.Patch(facecolor=PALETTE[c % len(PALETTE)],
                          label=f"Clade {c+1}", edgecolor="#1a2a3a")
           for c in unique_clades]
fig.legend(handles=handles, loc="lower center", ncol=min(len(handles), 10),
           frameon=False, labelcolor=TEXT, fontsize=8, bbox_to_anchor=(0.45, 0.0))

plt.savefig(str(out_path), dpi=180, bbox_inches="tight", facecolor=BG)
plt.close()
print(f"✓ Saved → {out_path}")


---
### Batch export — all 8 segments

In [ ]:
# ── Batch: export comparison PDF for every segment ──────────────────────────
# Uses umap_nn15_md0.1.parquet where available, falls back to first parquet found
# Set N_CLADES to your preferred cut

N_CLADES = 8
EMBED_PREFERENCE = "umap_nn15_md0.1"   # stem to prefer; falls back to first available

base = Path(PIPELINE)
out_dir = base / "output/phylo_umap_comparison"
out_dir.mkdir(parents=True, exist_ok=True)

for seg_num, seg_name in SEGMENT_NAMES.items():
    print(f"\n── Segment {seg_num} ({seg_name}) ────────────────────────────────────")

    tree    = load_tree(seg_num)
    dist_df = load_mldist(seg_num)
    if tree is None or dist_df is None:
        print("  ✗ Skipping — tree or dist missing"); continue

    clade_map = assign_clades(dist_df, N_CLADES)

    # pick embedding
    embeddings = get_available_embeddings(seg_num)
    path = next((v for k, v in embeddings.items() if EMBED_PREFERENCE in k), None)
    if path is None and embeddings:
        path = list(embeddings.values())[0]
    if path is None:
        print("  ✗ No embedding found — skipping"); continue

    emb_df = load_embedding(path)
    embed_stem = Path(path).stem
    stats = compute_stats(emb_df, clade_map, dist_df)
    if stats:
        print(f"  Mantel r={stats['mantel_r']:.3f}  ARI={stats['ARI']:.3f}  NMI={stats['NMI']:.3f}")

    fig = plt.figure(figsize=(20, 9), facecolor=BG)
    fig.suptitle(f"H5N1 · Segment {seg_num} ({seg_name}) · {N_CLADES} clades · {embed_stem}",
                 color=TEXT, fontsize=13, fontweight="bold")
    gs = fig.add_gridspec(1, 3, width_ratios=[2.2, 2.2, 1],
                          wspace=0.28, left=0.04, right=0.97, top=0.92, bottom=0.08)
    draw_tree(fig.add_subplot(gs[0]), tree, clade_map)
    draw_embedding(fig.add_subplot(gs[1]), emb_df, clade_map, embed_label=embed_stem)
    draw_stats(fig.add_subplot(gs[2]), stats, N_CLADES, seg_name,
               sum(1 for n in emb_df["name"] if n in clade_map), len(emb_df))

    unique_clades = sorted(set(clade_map.values()))
    handles = [mpatches.Patch(facecolor=PALETTE[c % len(PALETTE)],
                              label=f"Clade {c+1}", edgecolor="#1a2a3a")
               for c in unique_clades]
    fig.legend(handles=handles, loc="lower center", ncol=min(len(handles), 10),
               frameon=False, labelcolor=TEXT, fontsize=8, bbox_to_anchor=(0.45, 0.0))

    out_path = out_dir / f"segment_{seg_num}_{seg_name}_{embed_stem}_c{N_CLADES}.pdf"
    plt.savefig(str(out_path), dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close()
    print(f"  ✓ Saved → {out_path}")

print("\n✓ Batch export complete")
